# E-Commerce Customer & Revenue Analytics

## Data Quality Assessment and Cleaning

This notebook performs the initial data quality assessment and cleaning of the Olist Brazilian E-Commerce dataset.

### Objectives
- Understand the structure and grain of each dataset
- Identify missing values and duplicate records
- Validate data types and key relationships
- Investigate data quality issues before cleaning
- Prepare analytics-ready datasets for SQL analysis and Power BI

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

Matplotlib is building the font cache; this may take a moment.


In [2]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

In [3]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [4]:
customers.shape

(99441, 5)

In [5]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [6]:
print("Total rows:", len(customers))

print(
    "Unique customer_id:",
    customers["customer_id"].nunique()
)

print(
    "Unique customer_unique_id:",
    customers["customer_unique_id"].nunique()
)

Total rows: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096


In [7]:
customers["customer_unique_id"].value_counts().head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
dc813062e0fc23409cd255f7f53c7074     6
63cfc61cee11cbe306bff5857d00bfe4     6
f0e310a6839dce9de1638e0fe5ab282a     6
de34b16117594161a6a89c50b289d35a     6
Name: count, dtype: int64

In [8]:
repeat_customer = (
    customers["customer_unique_id"]
    .value_counts()
    .idxmax()
)

customers[
    customers["customer_unique_id"] == repeat_customer
]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
14186,1bd3585471932167ab72a84955ebefea,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
15321,a8fabc805e9a10a3c93ae5bff642b86b,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
16654,897b7f72042714efaa64ac306ba0cafc,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
36122,b2b13de0770e06de50080fea77c459e6,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
38073,42dbc1ad9d560637c9c4c1533746f86d,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
40141,dfb941d6f7b02f57a44c3b7c3fefb44b,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
48614,65f9db9dd07a4e79b625effa4c868fcb,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
52574,1c62b48fb34ee043310dcb233caabd2e,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
58707,a682769c4bc10fc6ef2101337a6c83c9,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
67996,6289b75219d757a56c0cce8d9e427900,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP


### Customer Table - Initial Findings

- The customer dataset contains one record per `customer_id`.
- `customer_id` is unique and acts as the customer identifier associated with an individual order.
- `customer_unique_id` identifies the same underlying customer across multiple purchases.
- Therefore, `customer_unique_id` will be used for customer-level behavioral analysis such as repeat purchases, retention, cohort analysis, and RFM segmentation.

1. Are any values missing?

In [9]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

2. Are there duplicate rows?

In [10]:
customers.duplicated().sum()

np.int64(0)

3. Is the supposed primary key actually unique?

In [11]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [15]:
print("Number of unique states:", customers["customer_state"].nunique())
print("Number of unique cities:", customers["customer_city"].nunique())

print("\nCustomer distribution by state:")
print(customers["customer_state"].value_counts())

Number of unique states: 27
Number of unique cities: 4119

Customer distribution by state:
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64


### Customer Table - Data Quality Findings

- The dataset contains 99,441 customer records representing 96,096 unique customers.
- `customer_id` contains no duplicates and can be used as the key for joining customers with orders.
- `customer_unique_id` is not unique because the same underlying customer may be associated with multiple order-specific customer IDs.
- No missing values or exact duplicate rows were identified.
- `customer_unique_id` will be used for customer-level behavioral analysis, including repeat purchases, retention, cohort analysis, and RFM segmentation.
- Customer location attributes can later support geographic analysis at the city and state levels.

In [16]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [17]:
print("Orders table shape:", orders.shape)

orders.info()

Orders table shape: (99441, 8)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [18]:
print("Total rows:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())
print("Unique customer IDs:", orders["customer_id"].nunique())

Total rows: 99441
Unique order IDs: 99441
Unique customer IDs: 99441


In [19]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [20]:
order_status_summary = (
    orders["order_status"]
    .value_counts()
    .reset_index()
)

order_status_summary.columns = ["order_status", "order_count"]

order_status_summary["percentage"] = (
    order_status_summary["order_count"]
    / len(orders)
    * 100
).round(2)

order_status_summary

,order_status,order_count,percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [21]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [22]:
missing_orders = pd.DataFrame({
    "missing_count": orders.isnull().sum(),
    "missing_percentage": (
        orders.isnull().mean() * 100
    ).round(2)
})

missing_orders

,missing_count,missing_percentage
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


In [23]:
pd.crosstab(
    orders["order_status"],
    orders["order_delivered_customer_date"].isnull(),
    margins=True
)

order_delivered_customer_date,False,True,All
order_status,,,
approved,0,2,2
canceled,6,619,625
created,0,5,5
delivered,96470,8,96478
invoiced,0,314,314
processing,0,301,301
shipped,0,1107,1107
unavailable,0,609,609
All,96476,2965,99441


In [24]:
delivered_missing_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isnull())
]

print(
    "Delivered orders with missing delivery date:",
    len(delivered_missing_date)
)

delivered_missing_date.head()

Delivered orders with missing delivery date: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00


In [25]:
pd.crosstab(
    orders["order_status"],
    orders["order_delivered_customer_date"].isnull(),
    margins=True
)

order_delivered_customer_date,False,True,All
order_status,,,
approved,0,2,2
canceled,6,619,625
created,0,5,5
delivered,96470,8,96478
invoiced,0,314,314
processing,0,301,301
shipped,0,1107,1107
unavailable,0,609,609
All,96476,2965,99441


In [26]:
delivered_missing_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isnull())
]

print(
    "Delivered orders with missing delivery date:",
    len(delivered_missing_date)
)

delivered_missing_date.head()

Delivered orders with missing delivery date: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00


In [27]:
pd.crosstab(
    orders["order_status"],
    orders["order_approved_at"].isnull(),
    margins=True
)

order_approved_at,False,True,All
order_status,,,
approved,2,0,2
canceled,484,141,625
created,0,5,5
delivered,96464,14,96478
invoiced,314,0,314
processing,301,0,301
shipped,1107,0,1107
unavailable,609,0,609
All,99281,160,99441


In [28]:
delivered_missing_approval = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_approved_at"].isnull())
]

print(
    "Delivered orders with missing approval date:",
    len(delivered_missing_approval)
)

delivered_missing_approval.head()

Delivered orders with missing approval date: 14


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00


In [29]:
pd.crosstab(
    orders["order_status"],
    orders["order_delivered_carrier_date"].isnull(),
    margins=True
)

order_delivered_carrier_date,False,True,All
order_status,,,
approved,0,2,2
canceled,75,550,625
created,0,5,5
delivered,96476,2,96478
invoiced,0,314,314
processing,0,301,301
shipped,1107,0,1107
unavailable,0,609,609
All,97658,1783,99441


In [30]:
delivered_missing_carrier = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_carrier_date"].isnull())
]

print(
    "Delivered orders with missing carrier date:",
    len(delivered_missing_carrier)
)

delivered_missing_carrier.head()

Delivered orders with missing carrier date: 2


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


In [31]:
print("Exact duplicate rows:", orders.duplicated().sum())

print(
    "Duplicate order IDs:",
    orders["order_id"].duplicated().sum()
)

Exact duplicate rows: 0
Duplicate order IDs: 0


In [32]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [33]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

In [34]:
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [35]:
orders[date_columns].isnull().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [36]:
approved_before_purchase = orders[
    orders["order_approved_at"]
    < orders["order_purchase_timestamp"]
]

print(
    "Orders approved before purchase:",
    len(approved_before_purchase)
)

Orders approved before purchase: 0


In [37]:
carrier_before_purchase = orders[
    orders["order_delivered_carrier_date"]
    < orders["order_purchase_timestamp"]
]

print(
    "Orders handed to carrier before purchase:",
    len(carrier_before_purchase)
)

Orders handed to carrier before purchase: 166


In [38]:
delivered_before_purchase = orders[
    orders["order_delivered_customer_date"]
    < orders["order_purchase_timestamp"]
]

print(
    "Orders delivered before purchase:",
    len(delivered_before_purchase)
)

Orders delivered before purchase: 0


In [39]:
delivered_before_carrier = orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
]

print(
    "Orders delivered before carrier handoff:",
    len(delivered_before_carrier)
)

Orders delivered before carrier handoff: 23


In [40]:
chronology_checks = pd.DataFrame({
    "Check": [
        "Approved before purchase",
        "Carrier handoff before purchase",
        "Delivered before purchase",
        "Delivered before carrier handoff"
    ],
    "Anomaly Count": [
        len(approved_before_purchase),
        len(carrier_before_purchase),
        len(delivered_before_purchase),
        len(delivered_before_carrier)
    ]
})

chronology_checks

,Check,Anomaly Count
0,Approved before purchase,0
1,Carrier handoff before purchase,166
2,Delivered before purchase,0
3,Delivered before carrier handoff,23


In [41]:
carrier_before_purchase_analysis = carrier_before_purchase.copy()

carrier_before_purchase_analysis["time_difference_hours"] = (
    carrier_before_purchase_analysis["order_purchase_timestamp"]
    - carrier_before_purchase_analysis["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

carrier_before_purchase_analysis[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "time_difference_hours"
    ]
].sort_values(
    "time_difference_hours",
    ascending=False
).head(10)

,order_id,order_purchase_timestamp,order_delivered_carrier_date,time_difference_hours
25883,7c48bb55e8e4f7e56d412e9653db37bc,2018-07-16 18:40:53,2018-01-26 13:35:00,4109.098056
83321,4021cd7611d6d9ce5ffcd24817fc374f,2018-08-18 11:49:40,2018-08-14 06:22:00,101.461111
67844,db090a16182b263b1e896bb26c6f66cf,2018-07-13 16:14:08,2018-07-13 13:59:00,2.252222
65452,9711d975b961355b4b5d636857e48498,2018-06-13 15:23:50,2018-06-13 13:15:00,2.147222
79401,89d32b64af005178b318f76cd60f2c3c,2018-07-06 11:54:40,2018-07-06 09:48:00,2.111111
13969,6192897f85cb2aff6dd1fca56fddb45e,2018-07-18 13:34:22,2018-07-18 11:33:00,2.022778
43657,3ed559dec69ab96d9074fedaf18650eb,2018-06-15 13:20:09,2018-06-15 11:22:00,1.969167
4256,4e157a36ea9cf89bde6fff57a780b525,2018-08-24 14:37:50,2018-08-24 12:43:00,1.913889
74503,119060af27b04f5fae8b2b0e27eef7f9,2018-06-11 14:34:47,2018-06-11 12:45:00,1.829722
21910,e23c9dcc304042fa90d538be679fa68d,2018-06-13 11:41:23,2018-06-13 09:53:00,1.806389


In [42]:
carrier_before_purchase_analysis[
    "time_difference_hours"
].describe()

count     166.000000
mean       26.039478
std       318.923845
min         0.006667
25%         0.234444
50%         0.599861
75%         1.017500
max      4109.098056
Name: time_difference_hours, dtype: float64

In [43]:
delivered_before_carrier_analysis = delivered_before_carrier.copy()

delivered_before_carrier_analysis["time_difference_hours"] = (
    delivered_before_carrier_analysis["order_delivered_carrier_date"]
    - delivered_before_carrier_analysis["order_delivered_customer_date"]
).dt.total_seconds() / 3600

delivered_before_carrier_analysis[
    [
        "order_id",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "time_difference_hours"
    ]
].sort_values(
    "time_difference_hours",
    ascending=False
).head(10)

,order_id,order_delivered_carrier_date,order_delivered_customer_date,time_difference_hours
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,2017-03-02 17:34:26,2017-02-14 15:15:57,386.308056
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,2017-08-09 18:18:43,2017-08-01 21:13:01,189.095000
45302,29941903985f944b0ffc49c479c1547d,2017-06-09 15:07:29,2017-06-02 11:09:16,171.970278
14474,dceb62e8fa94b46006c9554fed743df0,2017-08-01 18:23:30,2017-07-26 18:09:10,144.238889
49933,76458889992169d3135b264dc13aec67,2016-10-26 11:43:06,2016-10-20 18:03:17,137.663611
71227,19feb5627c41ea1b36a8e50a469b3644,2016-10-26 11:42:05,2016-10-20 19:07:54,136.569722
74967,d5558a097766363b8e76b38c43332e8a,2017-02-15 08:55:26,2017-02-10 07:58:32,120.948333
41636,b866af202be0692766081310cd4085e1,2017-02-20 02:32:08,2017-02-15 03:53:46,118.639444
6437,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-28 16:57:58,2017-07-25 19:32:56,69.417222
78556,ea1dcb4757a844d2642547797bd5feb0,2017-07-27 19:21:31,2017-07-25 19:43:10,47.639167


In [44]:
delivered_before_carrier_analysis[
    "time_difference_hours"
].describe()

count     23.000000
mean      78.455133
std       89.308070
min        0.388333
25%       23.338333
50%       39.864444
75%      128.759028
max      386.308056
Name: time_difference_hours, dtype: float64

### Chronological Data Quality Findings

- No orders were approved or delivered before their recorded purchase timestamp.
- 166 orders contained carrier handoff timestamps earlier than their purchase timestamp.
- 23 orders contained customer delivery timestamps earlier than their carrier handoff timestamp.
- These records were retained in the dataset because they may remain valid for sales, customer, and product analysis.
- Records with invalid chronological sequences will be excluded only from affected logistics and delivery-duration KPIs to prevent distortion of operational metrics.

In [45]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [46]:
orders[
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
].head(10)

,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.436574
1,2018-07-24 20:41:37,2018-08-07 15:27:45,13.782037
2,2018-08-08 08:38:49,2018-08-17 18:06:29,9.394213
3,2017-11-18 19:28:06,2017-12-02 00:28:42,13.208750
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2.873877
5,2017-07-09 21:57:05,2017-07-26 10:57:55,16.542245
6,2017-04-11 12:22:08,NaT,NaN
7,2017-05-16 13:10:30,2017-05-26 12:55:51,9.989826
8,2017-01-23 18:29:09,2017-02-02 14:08:10,9.818762
9,2017-07-29 11:55:02,2017-08-16 17:14:30,18.221852


In [47]:
print(
    "Negative delivery durations:",
    (orders["delivery_days"] < 0).sum()
)

orders["delivery_days"].describe()

Negative delivery durations: 0


count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_days, dtype: float64

In [48]:
orders["delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

In [49]:
orders[
    [
        "order_estimated_delivery_date",
        "order_delivered_customer_date",
        "delay_days"
    ]
].head(10)

,order_estimated_delivery_date,order_delivered_customer_date,delay_days
0,2017-10-18,2017-10-10 21:25:13,-7.107488
1,2018-08-13,2018-08-07 15:27:45,-5.355729
2,2018-09-04,2018-08-17 18:06:29,-17.245498
3,2017-12-15,2017-12-02 00:28:42,-12.980069
4,2018-02-26,2018-02-16 18:17:02,-9.238171
5,2017-08-01,2017-07-26 10:57:55,-5.543113
6,2017-05-09,NaT,NaN
7,2017-06-07,2017-05-26 12:55:51,-11.461215
8,2017-03-06,2017-02-02 14:08:10,-31.410995
9,2017-08-23,2017-08-16 17:14:30,-6.281597


In [50]:
orders["is_late"] = orders["delay_days"] > 0

In [51]:
orders["is_late"] = np.where(
    orders["delay_days"].isna(),
    np.nan,
    np.where(
        orders["delay_days"] > 0,
        1,
        0
    )
)

In [52]:
orders["is_late"].value_counts(dropna=False)

is_late
0.0    88649
1.0     7827
NaN     2965
Name: count, dtype: int64

In [53]:
orders["approval_time_hours"] = (
    orders["order_approved_at"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

In [54]:
orders["approval_time_hours"].describe()

count    99281.000000
mean        10.419094
std         26.038004
min          0.000000
25%          0.215000
50%          0.343333
75%         14.580833
max       4509.180556
Name: approval_time_hours, dtype: float64

In [55]:
print(
    "Negative approval durations:",
    (orders["approval_time_hours"] < 0).sum()
)

Negative approval durations: 0


In [56]:
orders["carrier_processing_days"] = (
    orders["order_delivered_carrier_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

In [57]:
orders.loc[
    orders["carrier_processing_days"] < 0,
    "carrier_processing_days"
] = np.nan

In [58]:
print(
    "Negative carrier processing durations:",
    (orders["carrier_processing_days"] < 0).sum()
)

orders["carrier_processing_days"].describe()

Negative carrier processing durations: 0


count    97492.000000
mean         3.241404
std          3.569041
min          0.000370
25%          1.132046
50%          2.208611
75%          4.074387
max        125.775521
Name: carrier_processing_days, dtype: float64

In [59]:
orders["shipping_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

In [60]:
orders.loc[
    orders["shipping_days"] < 0,
    "shipping_days"
] = np.nan

In [61]:
print(
    "Negative shipping durations:",
    (orders["shipping_days"] < 0).sum()
)

orders["shipping_days"].describe()

Negative shipping durations: 0


count    96452.000000
mean         9.333551
std          8.758825
min          0.000000
25%          4.101525
50%          7.100312
75%         12.030443
max        205.190972
Name: shipping_days, dtype: float64

### Order-Level Analytical Features

The following analytical features were derived from the order lifecycle timestamps:

- `delivery_days`: Total elapsed time between order purchase and customer delivery.
- `delay_days`: Difference between actual and estimated delivery dates. Positive values indicate late delivery, while negative values indicate early delivery.
- `is_late`: Flag identifying orders delivered after their estimated delivery date. Orders without a valid actual delivery date remain unclassified.
- `approval_time_hours`: Time elapsed between order purchase and approval.
- `carrier_processing_days`: Time elapsed between purchase and carrier handoff.
- `shipping_days`: Time elapsed between carrier handoff and customer delivery.

Invalid negative durations caused by previously identified chronological timestamp anomalies were converted to missing values only for the affected duration metrics. The underlying order records were retained for analyses where those timestamps are not required.

In [62]:
delivered_orders = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
].copy()

print("Total orders:", len(orders))
print("Eligible delivered orders:", len(delivered_orders))

Total orders: 99441
Eligible delivered orders: 96470


In [74]:
delivery_kpis = {
    "Total Delivered Orders": len(delivered_orders),

    "Average Delivery Days":
        delivered_orders["delivery_days"].mean(),

    "Median Delivery Days":
        delivered_orders["delivery_days"].median(),

    "Minimum Delivery Days":
        delivered_orders["delivery_days"].min(),

    "Maximum Delivery Days":
        delivered_orders["delivery_days"].max()
}

delivery_kpis

{'Total Delivered Orders': 96470,
 'Average Delivery Days': np.float64(12.558217098051976),
 'Median Delivery Days': np.float64(10.217476851851853),
 'Minimum Delivery Days': np.float64(0.5334143518518518),
 'Maximum Delivery Days': np.float64(209.6286111111111)}

In [64]:
delivery_status_counts = (
    delivered_orders["is_late"]
    .value_counts()
    .sort_index()
)

delivery_status_counts

is_late
0.0    88644
1.0     7826
Name: count, dtype: int64

In [65]:
on_time_rate = (
    (delivered_orders["is_late"] == 0).mean()
    * 100
)

late_rate = (
    (delivered_orders["is_late"] == 1).mean()
    * 100
)

print(f"On-Time Delivery Rate: {on_time_rate:.2f}%")
print(f"Late Delivery Rate: {late_rate:.2f}%")

On-Time Delivery Rate: 91.89%
Late Delivery Rate: 8.11%


In [66]:
delivered_orders["delay_days"].describe()

count    96470.000000
mean       -11.178126
std         10.184354
min       -146.016123
25%        -16.244065
50%        -11.948102
75%         -6.389815
max        188.975081
Name: delay_days, dtype: float64

In [67]:
late_orders = delivered_orders[
    delivered_orders["delay_days"] > 0
].copy()

print(
    "Average delay among late orders:",
    round(late_orders["delay_days"].mean(), 2),
    "days"
)

print(
    "Median delay among late orders:",
    round(late_orders["delay_days"].median(), 2),
    "days"
)

Average delay among late orders: 9.55 days
Median delay among late orders: 5.81 days


In [68]:
delivered_orders["delivery_days"].describe(
    percentiles=[0.90, 0.95, 0.99]
)

count    96470.000000
mean        12.558217
std          9.546156
min          0.533414
90%         23.096377
95%         29.274046
99%         46.050262
max        209.628611
Name: delivery_days, dtype: float64

In [69]:
Q1 = delivered_orders["delivery_days"].quantile(0.25)
Q3 = delivered_orders["delivery_days"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", round(Q1, 2))
print("Q3:", round(Q3, 2))
print("IQR:", round(IQR, 2))
print("Lower Bound:", round(lower_bound, 2))
print("Upper Bound:", round(upper_bound, 2))

Q1: 6.77
Q3: 15.72
IQR: 8.95
Lower Bound: -6.66
Upper Bound: 29.15


In [70]:
delivery_outliers = delivered_orders[
    (delivered_orders["delivery_days"] < lower_bound) |
    (delivered_orders["delivery_days"] > upper_bound)
]

print(
    "Number of delivery-time outliers:",
    len(delivery_outliers)
)

print(
    "Outlier percentage:",
    round(
        len(delivery_outliers)
        / len(delivered_orders)
        * 100,
        2
    ),
    "%"
)

Number of delivery-time outliers: 4896
Outlier percentage: 5.08 %


In [71]:
delivered_orders[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "delay_days"
    ]
].sort_values(
    "delivery_days",
    ascending=False
).head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days
19590,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-09-19 14:36:39,2017-03-22,209.628611,181.608785
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-09-19 23:24:07,2018-03-15,208.351759,188.975081
61610,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-09-19 15:12:50,2017-04-07,195.634016,165.633912
70307,2fb597c2f772eca01b1f5c561bf6cc7b,2017-03-08 18:09:02,2017-09-19 14:33:17,2017-04-17,194.850174,155.606447
89130,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-09-19 14:00:04,2017-04-06,194.633611,166.583380
38509,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-09-19 14:38:21,2017-04-11,194.049583,161.609965
11399,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-07-13 20:51:31,2018-01-19,191.463542,175.869109
81401,2fe324febf907e3ea3f2aa9650869fa5,2017-03-13 20:17:10,2017-09-19 17:00:07,2017-04-05,189.863160,167.708414
54480,2d7561026d542c8dbd8f0daeadf67a43,2017-03-15 11:24:27,2017-09-19 14:38:18,2017-04-13,188.134618,159.609931
68769,c27815f7e3dd0b926b58552628481575,2017-03-15 23:23:17,2017-09-19 17:14:25,2017-04-10,187.743843,162.718345


### Initial Delivery Performance Findings

- 96,470 delivered orders had valid customer delivery timestamps and were eligible for delivery-performance analysis.
- Orders took an average of approximately 12.56 days to reach customers, while the median delivery time was approximately 10.22 days.
- The higher mean relative to the median indicates that longer-delivery cases increase the overall average delivery time.
- Approximately 91.89% of eligible delivered orders were delivered on or before their estimated delivery date, while 8.11% were delivered late.
- Using the IQR method, approximately 5.08% of delivered orders were identified as statistical delivery-time outliers.
- These outliers were retained because unusually long delivery times may represent genuine operational issues rather than invalid data and will be investigated further using seller, geographic, product, and customer-review information.

In [75]:
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [76]:
print("Order Items shape:", order_items.shape)

order_items.info()

Order Items shape: (112650, 7)
<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [77]:
print("Total rows:", len(order_items))
print(
    "Unique order IDs:",
    order_items["order_id"].nunique()
)
print(
    "Unique product IDs:",
    order_items["product_id"].nunique()
)
print(
    "Unique seller IDs:",
    order_items["seller_id"].nunique()
)

Total rows: 112650
Unique order IDs: 98666
Unique product IDs: 32951
Unique seller IDs: 3095


In [78]:
items_per_order = (
    order_items
    .groupby("order_id")
    .size()
    .sort_values(ascending=False)
)

items_per_order.head(10)

order_id
8272b63d03f5f79c56e9e4120aec44ef    21
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
9ef13efd6949e4573a18964dd1bbe7f5    15
428a2f660dc84138d969ccd69a0ab6d5    15
9bdc4d4c71aa1de4606060929dee888c    14
73c8ab38f07dc94389065f7eba4f297a    14
37ee401157a3a0b28c9c6d0ed8c3b24b    13
2c2a19b5703863c908512d135aa6accc    12
c05d6a79e55da72ca780ce90364abed9    12
dtype: int64

In [79]:
print(
    "Maximum item rows in a single order:",
    items_per_order.max()
)

print(
    "Orders with more than one item row:",
    (items_per_order > 1).sum()
)

Maximum item rows in a single order: 21
Orders with more than one item row: 9803


In [80]:
largest_order_id = items_per_order.idxmax()

largest_order_id

'8272b63d03f5f79c56e9e4120aec44ef'

In [81]:
largest_order = order_items[
    order_items["order_id"] == largest_order_id
]

largest_order

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [82]:
print("Item rows:", len(largest_order))

print(
    "Unique products:",
    largest_order["product_id"].nunique()
)

print(
    "Unique sellers:",
    largest_order["seller_id"].nunique()
)

Item rows: 21
Unique products: 3
Unique sellers: 1


In [83]:
order_items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [84]:
print(
    "Exact duplicate rows:",
    order_items.duplicated().sum()
)

Exact duplicate rows: 0


In [85]:
print(
    "Duplicate order_id + order_item_id combinations:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Duplicate order_id + order_item_id combinations: 0


In [86]:
order_items[
    ["price", "freight_value"]
].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [87]:
print(
    "Negative prices:",
    (order_items["price"] < 0).sum()
)

print(
    "Zero prices:",
    (order_items["price"] == 0).sum()
)

print(
    "Negative freight values:",
    (order_items["freight_value"] < 0).sum()
)

print(
    "Zero freight values:",
    (order_items["freight_value"] == 0).sum()
)

Negative prices: 0
Zero prices: 0
Negative freight values: 0
Zero freight values: 383


In [88]:
order_items["price"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
50%          74.990000
75%         134.900000
90%         229.800000
95%         349.900000
99%         890.000000
max        6735.000000
Name: price, dtype: float64

In [89]:
order_items[
    [
        "order_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
].sort_values(
    "price",
    ascending=False
).head(10)

,order_id,product_id,seller_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,259037a6a41845e455183f89c5035f18,c72de06d72748d1a0dfb2125be43ba63,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,6cdf8fc1d741c76586d8b6b15e9eef30,ed4acab38528488b65a9a9c603ff024a,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,dd113cb02b2af9c8e5787e8f1f0722f6,821fb029fc6e495ca4f08a35d51e53a5,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,6902c1962dd19d540807d0ab8fade5c6,fa1c13f2614d7b5c4749cbc52fecda94,3999.90,17.01


In [90]:
payments = pd.read_csv(
    "../data/raw/olist_order_payments_dataset.csv"
)

payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [91]:
print("Payments table shape:", payments.shape)

payments.info()

Payments table shape: (103886, 5)
<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [92]:
print("Total payment rows:", len(payments))

print(
    "Unique order IDs:",
    payments["order_id"].nunique()
)

print(
    "Unique payment types:",
    payments["payment_type"].nunique()
)

Total payment rows: 103886
Unique order IDs: 99440
Unique payment types: 5


In [93]:
payments_per_order = (
    payments
    .groupby("order_id")
    .size()
    .sort_values(ascending=False)
)

payments_per_order.head(10)

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
ee9ca989fc93ba09a6eddc250ce01742    19
4bfcba9e084f46c8e3cb49b0fa6e6159    15
21577126c19bf11a0b91592e5844ba78    15
3c58bffb70dcf45f12bdf66a3c215905    14
4689b1816de42507a7d63a4617383c59    14
dtype: int64

In [94]:
print(
    "Maximum payment records for one order:",
    payments_per_order.max()
)

print(
    "Orders with multiple payment records:",
    (payments_per_order > 1).sum()
)

Maximum payment records for one order: 29
Orders with multiple payment records: 2961


In [95]:
most_payment_order = payments_per_order.idxmax()

payments[
    payments["order_id"] == most_payment_order
].sort_values("payment_sequential")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
82593,fa65dad1b0e818e3ccc5cb0e39231352,7,voucher,1,0.32
68853,fa65dad1b0e818e3ccc5cb0e39231352,8,voucher,1,26.02
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86


In [96]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [97]:
payment_type_summary = (
    payments["payment_type"]
    .value_counts()
    .reset_index()
)

payment_type_summary.columns = [
    "payment_type",
    "payment_count"
]

payment_type_summary["percentage"] = (
    payment_type_summary["payment_count"]
    / len(payments)
    * 100
).round(2)

payment_type_summary

,payment_type,payment_count,percentage
0,credit_card,76795,73.92
1,boleto,19784,19.04
2,voucher,5775,5.56
3,debit_card,1529,1.47
4,not_defined,3,0.00


In [98]:
print("Missing values:")
print(payments.isnull().sum())

print(
    "\nExact duplicate rows:",
    payments.duplicated().sum()
)

print(
    "\nNegative payment values:",
    (payments["payment_value"] < 0).sum()
)

print(
    "Zero payment values:",
    (payments["payment_value"] == 0).sum()
)

print(
    "Negative installments:",
    (payments["payment_installments"] < 0).sum()
)

print(
    "Zero installments:",
    (payments["payment_installments"] == 0).sum()
)

Missing values:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Exact duplicate rows: 0

Negative payment values: 0
Zero payment values: 9
Negative installments: 0
Zero installments: 2


In [99]:
payments["payment_value"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
50%         100.000000
75%         171.837500
90%         297.270000
95%         437.635000
99%        1039.916500
max       13664.080000
Name: payment_value, dtype: float64

In [100]:
payments[
    [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
].sort_values(
    "payment_value",
    ascending=False
).head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,1,boleto,1,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,1,boleto,1,6726.66
62409,2cc9089445046817a7539d90805e6e5a,1,boleto,1,6081.54
43232,a96610ab360d42a2e5335a3998b4718a,1,credit_card,10,4950.34
70320,b4c4b76c642808cbe472a32b86cddc95,1,credit_card,5,4809.44
6440,199af31afc78c699f0dbf71fb178d4d4,1,credit_card,8,4764.34
67546,8dbc85d1447242f3b127dda390d56e19,1,credit_card,8,4681.78


In [101]:
payments[
    payments["payment_value"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [102]:
payments[
    payments["payment_installments"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [103]:
order_item_financials = (
    order_items
    .groupby("order_id")
    .agg(
        merchandise_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

order_item_financials["gross_order_value"] = (
    order_item_financials["merchandise_value"]
    + order_item_financials["freight_value"]
)

order_item_financials.head()

,order_id,merchandise_value,freight_value,item_count,unique_products,unique_sellers,gross_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,1,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,1,1,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,1,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,1,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,1,1,218.04


In [104]:
print(
    "Rows in order-level item financials:",
    len(order_item_financials)
)

print(
    "Unique orders:",
    order_item_financials["order_id"].nunique()
)

Rows in order-level item financials: 98666
Unique orders: 98666


In [105]:
order_payment_financials = (
    payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_records=("payment_sequential", "count"),
        payment_methods=("payment_type", "nunique"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

order_payment_financials.head()

,order_id,total_payment_value,payment_records,payment_methods,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3


In [106]:
print(
    "Rows in order-level payment financials:",
    len(order_payment_financials)
)

print(
    "Unique orders:",
    order_payment_financials["order_id"].nunique()
)

Rows in order-level payment financials: 99440
Unique orders: 99440


In [107]:
financial_reconciliation = (
    order_item_financials
    .merge(
        order_payment_financials,
        on="order_id",
        how="inner"
    )
)

print(
    "Orders available for reconciliation:",
    len(financial_reconciliation)
)

financial_reconciliation.head()

Orders available for reconciliation: 98665


,order_id,merchandise_value,freight_value,item_count,unique_products,unique_sellers,gross_order_value,total_payment_value,payment_records,payment_methods,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,1,1,72.19,72.19,1,1,2
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,1,1,259.83,259.83,1,1,3
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,1,1,216.87,216.87,1,1,5
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,1,1,25.78,25.78,1,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,1,1,218.04,218.04,1,1,3


In [108]:
financial_reconciliation["payment_difference"] = (
    financial_reconciliation["total_payment_value"]
    - financial_reconciliation["gross_order_value"]
)

In [109]:
financial_reconciliation[
    [
        "gross_order_value",
        "total_payment_value",
        "payment_difference"
    ]
].describe()

,gross_order_value,total_payment_value,payment_difference
count,98665.000000,98665.000000,98665.000000
mean,160.577812,160.606904,0.029092
std,220.467197,220.484252,1.129221
min,9.590000,9.590000,-51.620000
25%,61.980000,62.000000,0.000000
50%,105.290000,105.290000,0.000000
75%,176.870000,176.880000,0.000000
max,13664.080000,13664.080000,182.810000


In [110]:
tolerance = 0.01

financial_reconciliation["payment_matches"] = (
    financial_reconciliation["payment_difference"].abs()
    <= tolerance
)

financial_reconciliation["payment_matches"].value_counts()

payment_matches
True     98284
False      381
Name: count, dtype: int64

In [111]:
match_rate = (
    financial_reconciliation["payment_matches"].mean()
    * 100
)

print(
    f"Payment reconciliation match rate: {match_rate:.2f}%"
)

Payment reconciliation match rate: 99.61%


In [112]:
payment_mismatches = (
    financial_reconciliation[
        ~financial_reconciliation["payment_matches"]
    ]
    .copy()
)

print(
    "Orders with payment mismatch:",
    len(payment_mismatches)
)

payment_mismatches[
    [
        "order_id",
        "merchandise_value",
        "freight_value",
        "gross_order_value",
        "total_payment_value",
        "payment_difference"
    ]
].sort_values(
    "payment_difference",
    key=abs,
    ascending=False
).head(20)

Orders with payment mismatch: 381


,order_id,merchandise_value,freight_value,gross_order_value,total_payment_value,payment_difference
79537,ce6d150fb29ada17d2082f4847107665,1299.00,104.66,1403.66,1586.47,182.81
42515,6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,287.91,406.92,119.01
43434,70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,466.93,578.82,111.89
58752,996c7e73600ad3723e8627ab7bef81e4,559.90,28.00,587.90,664.43,76.53
43437,70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,213.15,274.84,61.69
72496,bc2c82b0ef78d2252b6176d1972db7c9,165.00,77.01,242.01,303.02,61.01
67466,af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,413.17,466.97,53.80
14628,262118ce178bb3e4590a3adcf6d62e6b,119.80,57.94,177.74,126.12,-51.62
73899,bfdb5bbb06458d600a33d61f5f287472,297.00,51.93,348.93,394.36,45.43
54305,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.80,44.65,254.45,293.89,39.44


### Financial Data Modeling & Reconciliation

- The order-items dataset was aggregated from item-level to order-level grain, producing 98,666 unique order records.
- The payments dataset was independently aggregated to order-level grain, producing 99,440 unique order records.
- Aggregating both datasets before joining prevents many-to-many joins and potential inflation of financial metrics.
- 98,665 orders were available in both datasets for financial reconciliation.
- Approximately 99.61% of matched orders had total payment values within ±0.01 of the sum of merchandise and freight values.
- 381 orders showed larger differences between recorded payment value and calculated gross order value and were retained for further analysis rather than automatically classified as data errors.
- Financial metrics will distinguish between merchandise value, freight value, gross order value, and recorded payment value to maintain consistent KPI definitions.

In [113]:
products = pd.read_csv(
    "../data/raw/olist_products_dataset.csv"
)

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [114]:
print("Total product rows:", len(products))

print(
    "Unique product IDs:",
    products["product_id"].nunique()
)

print(
    "Unique categories:",
    products["product_category_name"].nunique()
)

Total product rows: 32951
Unique product IDs: 32951
Unique categories: 73


In [115]:
print(
    "Exact duplicate rows:",
    products.duplicated().sum()
)

print(
    "Duplicate product IDs:",
    products["product_id"].duplicated().sum()
)

print("\nMissing values:")

print(
    products.isnull().sum()
)

Exact duplicate rows: 0
Duplicate product IDs: 0

Missing values:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [116]:
products[
    "product_category_name"
].value_counts().head(20)

product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
perfumaria                            868
papelaria                             849
fashion_bolsas_e_acessorios           849
cool_stuff                            789
ferramentas_jardim                    753
pet_shop                              719
eletronicos                           517
construcao_ferramentas_construcao     400
eletrodomesticos                      370
Name: count, dtype: int64

In [117]:
category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

category_translation.head(10)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


In [118]:
print(
    "Translation rows:",
    len(category_translation)
)

print(
    "Unique Portuguese categories:",
    category_translation[
        "product_category_name"
    ].nunique()
)

print(
    "Unique English categories:",
    category_translation[
        "product_category_name_english"
    ].nunique()
)

Translation rows: 71
Unique Portuguese categories: 71
Unique English categories: 71


In [119]:
products_en = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

products_en[
    [
        "product_id",
        "product_category_name",
        "product_category_name_english"
    ]
].head(10)

,product_id,product_category_name,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares
5,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments
6,732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff
7,2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor
8,37cc742be07708b53a98702e77a21a02,eletrodomesticos,home_appliances
9,8c92109888e8cdf9d66dc7e463025574,brinquedos,toys


In [120]:
print(
    "Products without English category:",
    products_en[
        "product_category_name_english"
    ].isnull().sum()
)

Products without English category: 623


In [123]:
missing_original_category = products_en[
    "product_category_name"
].isnull().sum()

missing_translation = (
    products_en["product_category_name"].notna()
    &
    products_en["product_category_name_english"].isna()
)

print(
    "Products with missing original category:",
    missing_original_category
)

print(
    "Products with category but no English translation:",
    missing_translation.sum()
)

Products with missing original category: 610
Products with category but no English translation: 13


In [124]:
products_en["category_name"] = (
    products_en["product_category_name_english"]
    .fillna(products_en["product_category_name"])
    .fillna("unknown")
)

In [125]:
print(
    "Missing values in final category_name:",
    products_en["category_name"].isnull().sum()
)

print(
    "Products classified as unknown:",
    (products_en["category_name"] == "unknown").sum()
)

print(
    "Final number of categories:",
    products_en["category_name"].nunique()
)

Missing values in final category_name: 0
Products classified as unknown: 610
Final number of categories: 74


In [126]:
order_items_enriched = order_items.merge(
    products_en[
        [
            "product_id",
            "category_name"
        ]
    ],
    on="product_id",
    how="left"
)

print("Original order-item rows:", len(order_items))
print("Enriched order-item rows:", len(order_items_enriched))

print(
    "Missing category after join:",
    order_items_enriched["category_name"].isnull().sum()
)

Original order-item rows: 112650
Enriched order-item rows: 112650
Missing category after join: 0


In [127]:
category_performance = (
    order_items_enriched
    .groupby("category_name")
    .agg(
        item_rows=("order_item_id", "count"),
        unique_orders=("order_id", "nunique"),
        unique_products=("product_id", "nunique"),
        merchandise_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

category_performance["gross_order_value"] = (
    category_performance["merchandise_value"]
    + category_performance["freight_value"]
)

category_performance["avg_item_price"] = (
    category_performance["merchandise_value"]
    / category_performance["item_rows"]
)

category_performance.head()

,category_name,item_rows,unique_orders,unique_products,merchandise_value,freight_value,gross_order_value,avg_item_price
0,agro_industry_and_commerce,212,182,74,72530.47,5843.60,78374.07,342.124858
1,air_conditioning,297,253,124,55024.96,6749.23,61774.19,185.269226
2,art,209,202,55,24202.64,4045.17,28247.81,115.802105
3,arts_and_craftmanship,24,23,19,1814.01,370.13,2184.14,75.583750
4,audio,364,350,58,50688.50,5710.44,56398.94,139.254121


In [128]:
category_performance[
    [
        "category_name",
        "unique_orders",
        "merchandise_value",
        "gross_order_value",
        "avg_item_price"
    ]
].sort_values(
    "merchandise_value",
    ascending=False
).head(10)

,category_name,unique_orders,merchandise_value,gross_order_value,avg_item_price
43,health_beauty,8836,1258681.34,1441248.07,130.163531
73,watches_gifts,5624,1205005.68,1305541.61,201.135984
7,bed_bath_table,9417,1036988.68,1241681.72,93.296327
67,sports_leisure,7720,988048.97,1156656.48,114.344285
15,computers_accessories,6689,911954.32,1059272.40,116.513903
39,furniture_decor,6449,729762.49,902511.79,87.564494
20,cool_stuff,3632,635290.85,719329.95,167.357969
49,housewares,5884,632248.66,778397.77,90.788148
5,auto,3897,592720.11,685384.32,139.957523
42,garden_tools,3518,485256.46,584219.21,111.630196


In [129]:
category_performance[
    [
        "category_name",
        "unique_orders",
        "merchandise_value",
        "avg_item_price"
    ]
].sort_values(
    "unique_orders",
    ascending=False
).head(10)

,category_name,unique_orders,merchandise_value,avg_item_price
7,bed_bath_table,9417,1036988.68,93.296327
43,health_beauty,8836,1258681.34,130.163531
67,sports_leisure,7720,988048.97,114.344285
15,computers_accessories,6689,911954.32,116.513903
39,furniture_decor,6449,729762.49,87.564494
49,housewares,5884,632248.66,90.788148
73,watches_gifts,5624,1205005.68,201.135984
70,telephony,4199,323667.53,71.213978
5,auto,3897,592720.11,139.957523
71,toys,3886,483946.60,117.548360


In [130]:
total_merchandise_value = (
    category_performance["merchandise_value"].sum()
)

category_performance["merchandise_share_pct"] = (
    category_performance["merchandise_value"]
    / total_merchandise_value
    * 100
)

In [131]:
category_performance[
    [
        "category_name",
        "merchandise_value",
        "merchandise_share_pct"
    ]
].sort_values(
    "merchandise_value",
    ascending=False
).head(10)

,category_name,merchandise_value,merchandise_share_pct
43,health_beauty,1258681.34,9.260700
73,watches_gifts,1205005.68,8.865783
7,bed_bath_table,1036988.68,7.629605
67,sports_leisure,988048.97,7.269533
15,computers_accessories,911954.32,6.709669
39,furniture_decor,729762.49,5.369200
20,cool_stuff,635290.85,4.674128
49,housewares,632248.66,4.651745
5,auto,592720.11,4.360916
42,garden_tools,485256.46,3.570256


In [132]:
reviews = pd.read_csv(
    "../data/raw/olist_order_reviews_dataset.csv"
)

reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [133]:
print("Total review rows:", len(reviews))

print(
    "Unique review IDs:",
    reviews["review_id"].nunique()
)

print(
    "Unique order IDs:",
    reviews["order_id"].nunique()
)

print(
    "Duplicate review IDs:",
    reviews["review_id"].duplicated().sum()
)

print(
    "Orders appearing more than once:",
    reviews["order_id"].duplicated().sum()
)

Total review rows: 99224
Unique review IDs: 98410
Unique order IDs: 98673
Duplicate review IDs: 814
Orders appearing more than once: 551


In [134]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [135]:
reviews["review_score"].describe()

count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

In [136]:
print(
    "Invalid review scores:",
    (~reviews["review_score"].between(1, 5)).sum()
)

Invalid review scores: 0


In [137]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [138]:
reviews["has_comment"] = (
    reviews["review_comment_message"].notna()
).astype(int)

In [139]:
reviews["review_sentiment_group"] = np.select(
    [
        reviews["review_score"] <= 2,
        reviews["review_score"] == 3,
        reviews["review_score"] >= 4
    ],
    [
        "Negative",
        "Neutral",
        "Positive"
    ],
    default="Unknown"
)

In [140]:
reviews["review_sentiment_group"].value_counts()

review_sentiment_group
Positive    76470
Negative    14575
Neutral      8179
Name: count, dtype: int64

In [141]:
print(
    "Average Review Score:",
    round(reviews["review_score"].mean(), 2)
)

positive_review_rate = (
    (reviews["review_score"] >= 4).mean()
    * 100
)

negative_review_rate = (
    (reviews["review_score"] <= 2).mean()
    * 100
)

print(
    f"Positive Review Rate: {positive_review_rate:.2f}%"
)

print(
    f"Negative Review Rate: {negative_review_rate:.2f}%"
)

Average Review Score: 4.09
Positive Review Rate: 77.07%
Negative Review Rate: 14.69%


In [142]:
review_counts_per_order = (
    reviews
    .groupby("order_id")
    .size()
    .sort_values(ascending=False)
)

review_counts_per_order.head(10)

order_id
8e17072ec97ce29f0e1f111e598b0c85    3
c88b1d1b157a9999ce368f218a407141    3
03c939fd7fd3b38f8485a0f95798f1f6    3
df56136b8031ecd28e200bb18e6ddb2e    3
29062384ce4975f78aeba6a496510386    2
fd95ae805c63c534f1a64589e102225e    2
8b3c2785144e72ccba9b0213f0f1cd1e    2
82fd1196a459f594fb1d66e667fc74c4    2
ca263afd88a8a1200605adbd4b63cd7d    2
4703440eb9289d3769819920e98ec061    2
dtype: int64

In [143]:
print(
    "Maximum review rows for one order:",
    review_counts_per_order.max()
)

print(
    "Orders with multiple review rows:",
    (review_counts_per_order > 1).sum()
)

Maximum review rows for one order: 3
Orders with multiple review rows: 547


In [144]:
most_reviewed_order = review_counts_per_order.idxmax()

reviews[
    reviews["order_id"] == most_reviewed_order
].sort_values("review_answer_timestamp")

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_comment,review_sentiment_group
64510,2d6ac45f859465b5c185274a1c929637,8e17072ec97ce29f0e1f111e598b0c85,1,NaN,Comprei 3 unidades do produto vieram 2 unidade...,2018-04-07 00:00:00,2018-04-07 21:13:05,1,Negative
44694,67c2557eb0bd72e3ece1e03477c9dff5,8e17072ec97ce29f0e1f111e598b0c85,1,NaN,Entregou o produto errado.,2018-04-07 00:00:00,2018-04-08 22:48:27,1,Negative
92300,6e4c4086d9611ae4cc0cc65a262751fe,8e17072ec97ce29f0e1f111e598b0c85,1,NaN,"Embora tenha entregue dentro do prazo, não env...",2018-04-14 00:00:00,2018-04-16 11:37:31,1,Negative


In [145]:
review_score_variation = (
    reviews
    .groupby("order_id")["review_score"]
    .nunique()
)

print(
    "Multi-review orders with different review scores:",
    (review_score_variation > 1).sum()
)

Multi-review orders with different review scores: 202


In [146]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    reviews[column] = pd.to_datetime(
        reviews[column],
        errors="coerce"
    )

reviews[review_date_columns].dtypes

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [147]:
reviews_sorted = reviews.sort_values(
    ["order_id", "review_answer_timestamp"]
)

In [148]:
order_reviews = (
    reviews_sorted
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    .copy()
)

print("Raw review rows:", len(reviews))
print("Order-level review rows:", len(order_reviews))
print(
    "Unique orders:",
    order_reviews["order_id"].nunique()
)

Raw review rows: 99224
Order-level review rows: 98673
Unique orders: 98673


In [149]:
delivery_reviews = delivered_orders.merge(
    order_reviews[
        [
            "order_id",
            "review_score"
        ]
    ],
    on="order_id",
    how="inner"
)

print(
    "Delivered orders with review data:",
    len(delivery_reviews)
)

print(
    "Unique orders after join:",
    delivery_reviews["order_id"].nunique()
)

Delivered orders with review data: 95824
Unique orders after join: 95824


In [150]:
late_vs_ontime_reviews = (
    delivery_reviews
    .groupby("is_late")
    .agg(
        orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        median_review_score=("review_score", "median")
    )
    .reset_index()
)

late_vs_ontime_reviews

,is_late,orders,avg_review_score,median_review_score
0,0.0,88163,4.294114,5.0
1,1.0,7661,2.565070,2.0


In [151]:
review_impact = (
    delivery_reviews
    .groupby("is_late")
    .agg(
        total_orders=("order_id", "nunique"),

        positive_review_rate=(
            "review_score",
            lambda x: (x >= 4).mean() * 100
        ),

        negative_review_rate=(
            "review_score",
            lambda x: (x <= 2).mean() * 100
        )
    )
    .reset_index()
)

review_impact

,is_late,total_orders,positive_review_rate,negative_review_rate
0,0.0,88163,82.788698,9.221556
1,1.0,7661,34.551625,54.066049


In [152]:
delivery_reviews["delivery_status_group"] = pd.cut(
    delivery_reviews["delay_days"],
    bins=[
        -float("inf"),
        0,
        3,
        7,
        14,
        float("inf")
    ],
    labels=[
        "On Time / Early",
        "1-3 Days Late",
        "4-7 Days Late",
        "8-14 Days Late",
        "15+ Days Late"
    ]
)

In [153]:
delay_satisfaction = (
    delivery_reviews
    .groupby(
        "delivery_status_group",
        observed=True
    )
    .agg(
        orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),

        positive_review_rate=(
            "review_score",
            lambda x: (x >= 4).mean() * 100
        ),

        negative_review_rate=(
            "review_score",
            lambda x: (x <= 2).mean() * 100
        )
    )
    .reset_index()
)

delay_satisfaction.round(2)

,delivery_status_group,orders,avg_review_score,positive_review_rate,negative_review_rate
0,On Time / Early,88163,4.29,82.79,9.22
1,1-3 Days Late,2636,3.77,66.27,19.12
2,4-7 Days Late,1773,2.32,27.86,61.31
3,8-14 Days Late,1748,1.74,12.70,78.15
4,15+ Days Late,1504,1.71,12.23,78.79


### Delivery Performance vs Customer Satisfaction

A strong association was observed between delivery delays and customer satisfaction:

- Orders delivered on time or early achieved an average review score of 4.29/5, with 82.79% positive reviews and only 9.22% negative reviews.
- Even short delays of 1–3 days reduced the average review score to 3.77 and increased the negative review rate to 19.12%.
- Customer satisfaction deteriorated sharply once delays exceeded 3 days. Orders delivered 4–7 days late received an average rating of only 2.32, with 61.31% negative reviews.
- For delays exceeding 8 days, average ratings fell below 1.75/5 and approximately 78%–79% of reviews were negative.
- The results indicate a strong negative association between delivery delays and customer satisfaction, suggesting that delivery reliability is an important operational factor associated with customer experience.

These findings establish delivery performance as a key dimension for further analysis across sellers, product categories, and geographic regions.

In [154]:
customer_orders = orders.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

print("Original order rows:", len(orders))
print("Customer-order rows:", len(customer_orders))

print(
    "Unique orders after join:",
    customer_orders["order_id"].nunique()
)

print(
    "Missing customer_unique_id:",
    customer_orders["customer_unique_id"].isnull().sum()
)

Original order rows: 99441
Customer-order rows: 99441
Unique orders after join: 99441
Missing customer_unique_id: 0


In [155]:
orders_per_customer = (
    customer_orders
    .groupby("customer_unique_id")["order_id"]
    .nunique()
    .sort_values(ascending=False)
)

orders_per_customer.describe()

count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: order_id, dtype: float64

In [156]:
repeat_customers = (
    orders_per_customer > 1
).sum()

one_time_customers = (
    orders_per_customer == 1
).sum()

total_customers = len(orders_per_customer)

repeat_customer_rate = (
    repeat_customers
    / total_customers
    * 100
)

print("Total unique customers:", total_customers)

print(
    "One-time customers:",
    one_time_customers
)

print(
    "Repeat customers:",
    repeat_customers
)

print(
    f"Repeat Customer Rate: {repeat_customer_rate:.2f}%"
)

print(
    "Maximum orders by one customer:",
    orders_per_customer.max()
)

Total unique customers: 96096
One-time customers: 93099
Repeat customers: 2997
Repeat Customer Rate: 3.12%
Maximum orders by one customer: 17


In [157]:
customer_frequency_distribution = (
    orders_per_customer
    .value_counts()
    .sort_index()
    .reset_index()
)

customer_frequency_distribution.columns = [
    "number_of_orders",
    "number_of_customers"
]

customer_frequency_distribution.head(10)

,number_of_orders,number_of_customers
0,1,93099
1,2,2745
2,3,203
3,4,30
4,5,8
5,6,6
6,7,3
7,9,1
8,17,1


In [158]:
customer_frequency_distribution[
    "customer_percentage"
] = (
    customer_frequency_distribution[
        "number_of_customers"
    ]
    / total_customers
    * 100
)

customer_frequency_distribution.head(10).round(2)

,number_of_orders,number_of_customers,customer_percentage
0,1,93099,96.88
1,2,2745,2.86
2,3,203,0.21
3,4,30,0.03
4,5,8,0.01
5,6,6,0.01
6,7,3,0.00
7,9,1,0.00
8,17,1,0.00


In [159]:
customer_state_summary = (
    customers
    .groupby("customer_state")
    .agg(
        unique_customers=(
            "customer_unique_id",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "unique_customers",
        ascending=False
    )
)

customer_state_summary.head(10)

,customer_state,unique_customers
25,SP,40302
18,RJ,12384
10,MG,11259
22,RS,5277
17,PR,4882
23,SC,3534
4,BA,3277
6,DF,2075
7,ES,1964
8,GO,1952


In [160]:
customer_state_summary[
    "customer_share_pct"
] = (
    customer_state_summary[
        "unique_customers"
    ]
    / customers[
        "customer_unique_id"
    ].nunique()
    * 100
)

customer_state_summary.head(10).round(2)

,customer_state,unique_customers,customer_share_pct
25,SP,40302,41.94
18,RJ,12384,12.89
10,MG,11259,11.72
22,RS,5277,5.49
17,PR,4882,5.08
23,SC,3534,3.68
4,BA,3277,3.41
6,DF,2075,2.16
7,ES,1964,2.04
8,GO,1952,2.03


In [161]:
customer_city_summary = (
    customers
    .groupby(
        [
            "customer_state",
            "customer_city"
        ]
    )
    .agg(
        unique_customers=(
            "customer_unique_id",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "unique_customers",
        ascending=False
    )
)

customer_city_summary.head(15)

,customer_state,customer_city,unique_customers
4176,SP,sao paulo,14984
2788,RJ,rio de janeiro,6620
1062,MG,belo horizonte,2672
601,DF,brasilia,2069
2406,PR,curitiba,1465
3729,SP,campinas,1398
3208,RS,porto alegre,1326
372,BA,salvador,1209
3836,SP,guarulhos,1153
4160,SP,sao bernardo do campo,908


In [162]:
customer_order_financials = (
    customer_orders
    .merge(
        order_item_financials[
            [
                "order_id",
                "merchandise_value",
                "freight_value",
                "gross_order_value",
                "item_count"
            ]
        ],
        on="order_id",
        how="left"
    )
)

print(
    "Rows after financial join:",
    len(customer_order_financials)
)

print(
    "Unique orders:",
    customer_order_financials[
        "order_id"
    ].nunique()
)

Rows after financial join: 99441
Unique orders: 99441


In [163]:
print(
    "Orders without item financials:",
    customer_order_financials[
        "gross_order_value"
    ].isnull().sum()
)

Orders without item financials: 775


In [164]:
state_performance = (
    customer_order_financials
    .groupby("customer_state")
    .agg(
        unique_customers=(
            "customer_unique_id",
            "nunique"
        ),

        orders=(
            "order_id",
            "nunique"
        ),

        merchandise_value=(
            "merchandise_value",
            "sum"
        ),

        freight_value=(
            "freight_value",
            "sum"
        ),

        gross_order_value=(
            "gross_order_value",
            "sum"
        )
    )
    .reset_index()
)

In [165]:
state_performance[
    "avg_gross_order_value"
] = (
    state_performance[
        "gross_order_value"
    ]
    / state_performance[
        "orders"
    ]
)

In [166]:
state_performance[
    "merchandise_share_pct"
] = (
    state_performance[
        "merchandise_value"
    ]
    / state_performance[
        "merchandise_value"
    ].sum()
    * 100
)

In [167]:
state_performance[
    [
        "customer_state",
        "unique_customers",
        "orders",
        "merchandise_value",
        "avg_gross_order_value",
        "merchandise_share_pct"
    ]
].sort_values(
    "merchandise_value",
    ascending=False
).head(10).round(2)

,customer_state,unique_customers,orders,merchandise_value,avg_gross_order_value,merchandise_share_pct
25,SP,40302,41746,5202955.05,141.85,38.28
18,RJ,12384,12852,1824092.67,165.71,13.42
10,MG,11259,11635,1585308.03,159.53,11.66
22,RS,5277,5466,750304.02,162.06,5.52
17,PR,4882,5045,683083.76,158.76,5.03
23,SC,3534,3637,520553.34,167.78,3.83
4,BA,3277,3380,511349.99,180.92,3.76
6,DF,2075,2140,302603.94,165.06,2.23
8,GO,1952,2020,294591.95,172.13,2.17
7,ES,1964,2033,275037.31,159.76,2.02


### Customer Purchase Behavior Findings

- The dataset contains 96,096 unique customers based on `customer_unique_id`.
- 93,099 customers placed only one observed order, while 2,997 customers placed multiple orders.
- The observed repeat customer rate is approximately 3.12%, indicating that repeat purchasing is relatively limited within the available observation period.
- The most frequent customer placed 17 orders.
- One-time customers should not automatically be classified as churned because customer activity is only observable within the dataset's available time period.
- 775 orders did not have corresponding item-level financial records and were retained rather than automatically excluded from non-financial analyses.

In [168]:
print(
    "Earliest purchase:",
    customer_order_financials[
        "order_purchase_timestamp"
    ].min()
)

print(
    "Latest purchase:",
    customer_order_financials[
        "order_purchase_timestamp"
    ].max()
)

Earliest purchase: 2016-09-04 21:15:19
Latest purchase: 2018-10-17 17:30:18


In [169]:
snapshot_date = (
    customer_order_financials[
        "order_purchase_timestamp"
    ].max()
    + pd.Timedelta(days=1)
)

print("RFM Snapshot Date:", snapshot_date)

RFM Snapshot Date: 2018-10-18 17:30:18


In [170]:
rfm_source = customer_order_financials[
    customer_order_financials[
        "merchandise_value"
    ].notna()
].copy()

In [171]:
rfm = (
    rfm_source
    .groupby("customer_unique_id")
    .agg(
        last_purchase=(
            "order_purchase_timestamp",
            "max"
        ),

        frequency=(
            "order_id",
            "nunique"
        ),

        monetary=(
            "merchandise_value",
            "sum"
        )
    )
    .reset_index()
)

In [172]:
rfm["recency"] = (
    snapshot_date
    - rfm["last_purchase"]
).dt.days

In [173]:
rfm = rfm[
    [
        "customer_unique_id",
        "recency",
        "frequency",
        "monetary",
        "last_purchase"
    ]
]

rfm.head()

,customer_unique_id,recency,frequency,monetary,last_purchase
0,0000366f3b9a7992bf8c76cfdf3221e2,161,1,129.90,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,18.90,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,586,1,69.00,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,370,1,25.99,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,337,1,180.00,2017-11-14 19:45:42


In [174]:
print("Customers in RFM:", len(rfm))

print("\nMissing values:")
print(rfm.isnull().sum())

print("\nRFM Summary:")
print(
    rfm[
        [
            "recency",
            "frequency",
            "monetary"
        ]
    ].describe()
)

Customers in RFM: 95420

Missing values:
customer_unique_id    0
recency               0
frequency             0
monetary              0
last_purchase         0
dtype: int64

RFM Summary:
            recency     frequency      monetary
count  95420.000000  95420.000000  95420.000000
mean     288.128118      1.034018    142.440198
std      153.157768      0.211234    217.656355
min       45.000000      1.000000      0.850000
25%      164.000000      1.000000     47.900000
50%      269.000000      1.000000     89.900000
75%      397.000000      1.000000    155.000000
max      773.000000     16.000000  13440.000000


In [175]:
print(
    "Customers with frequency > 1:",
    (rfm["frequency"] > 1).sum()
)

print(
    "Maximum frequency:",
    rfm["frequency"].max()
)

print(
    "Zero or negative monetary values:",
    (rfm["monetary"] <= 0).sum()
)

Customers with frequency > 1: 2913
Maximum frequency: 16
Zero or negative monetary values: 0


In [177]:
frequency_distribution = (
    rfm["frequency"]
    .value_counts()
    .sort_index()
    .reset_index()
)

frequency_distribution.columns = [
    "frequency",
    "customers"
]

frequency_distribution["percentage"] = (
    frequency_distribution["customers"]
    / len(rfm)
    * 100
)

frequency_distribution.head(15).round(2)

,frequency,customers,percentage
0,1,92507,96.95
1,2,2673,2.80
2,3,192,0.20
3,4,29,0.03
4,5,9,0.01
5,6,5,0.01
6,7,3,0.00
7,9,1,0.00
8,16,1,0.00


In [178]:
rfm["recency"].describe(
    percentiles=[
        0.20,
        0.40,
        0.60,
        0.80
    ]
)

count    95420.000000
mean       288.128118
std        153.157768
min         45.000000
20%        142.000000
40%        227.000000
60%        318.000000
80%        433.000000
max        773.000000
Name: recency, dtype: float64

In [179]:
rfm["monetary"].describe(
    percentiles=[
        0.20,
        0.40,
        0.50,
        0.60,
        0.80,
        0.90,
        0.95,
        0.99
    ]
)

count    95420.000000
mean       142.440198
std        217.656355
min          0.850000
20%         39.900000
40%         69.900000
50%         89.900000
60%        109.900000
80%        179.900000
90%        284.000000
95%        422.000000
99%       1013.593400
max      13440.000000
Name: monetary, dtype: float64

In [180]:
rfm["R_score"] = pd.qcut(
    rfm["recency"],
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

In [181]:
rfm.groupby("R_score").agg(
    customers=("customer_unique_id", "count"),
    min_recency=("recency", "min"),
    max_recency=("recency", "max")
)

,customers,min_recency,max_recency
R_score,,,
1,19040,434,773
2,18936,319,433
3,19126,228,318
4,19189,143,227
5,19129,45,142


In [182]:
rfm["M_score"] = pd.qcut(
    rfm["monetary"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [183]:
rfm.groupby("M_score").agg(
    customers=("customer_unique_id", "count"),
    min_monetary=("monetary", "min"),
    max_monetary=("monetary", "max"),
    avg_monetary=("monetary", "mean")
).round(2)

,customers,min_monetary,max_monetary,avg_monetary
M_score,,,,
1,19728,0.85,39.9,26.08
2,19815,39.90,69.9,55.14
3,17792,69.94,109.9,90.14
4,19051,109.95,179.9,140.43
5,19034,179.91,13440.0,404.81


In [184]:
def frequency_score(x):
    if x == 1:
        return 1
    elif x == 2:
        return 2
    elif x == 3:
        return 3
    elif x == 4:
        return 4
    else:
        return 5

rfm["F_score"] = (
    rfm["frequency"]
    .apply(frequency_score)
)

In [185]:
rfm.groupby("F_score").agg(
    customers=("customer_unique_id", "count"),
    min_frequency=("frequency", "min"),
    max_frequency=("frequency", "max")
)

,customers,min_frequency,max_frequency
F_score,,,
1,92507,1,1
2,2673,2,2
3,192,3,3
4,29,4,4
5,19,5,16


In [186]:
rfm["RFM_score"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

In [187]:
rfm[
    [
        "recency",
        "frequency",
        "monetary",
        "R_score",
        "F_score",
        "M_score",
        "RFM_score"
    ]
].head(10)

,recency,frequency,monetary,R_score,F_score,M_score,RFM_score
0,161,1,129.90,4,1,4,414
1,164,1,18.90,4,1,1,411
2,586,1,69.00,1,1,2,112
3,370,1,25.99,2,1,1,211
4,337,1,180.00,2,1,5,215
5,195,1,154.00,4,1,4,414
6,181,1,27.99,4,1,1,411
7,232,1,382.00,3,1,5,315
8,592,1,135.00,1,1,4,114
9,220,1,104.90,4,1,3,413


In [188]:
rfm[
    [
        "R_score",
        "F_score",
        "M_score"
    ]
].describe()

,R_score,F_score,M_score
count,95420.000000,95420.000000,95420.000000
mean,3.004517,1.033746,2.977447
std,1.414069,0.202100,1.425387
min,1.000000,1.000000,1.000000
25%,2.000000,1.000000,2.000000
50%,3.000000,1.000000,3.000000
75%,4.000000,1.000000,4.000000
max,5.000000,5.000000,5.000000


In [189]:
def assign_customer_segment(row):

    R = row["R_score"]
    F = row["F_score"]
    M = row["M_score"]

    # Repeat customers
    if F >= 2:

        if R >= 4 and M >= 4:
            return "Champions"

        elif R >= 4:
            return "Recent Repeat Buyers"

        else:
            return "At-Risk Repeat Buyers"

    # One-time customers
    else:

        if R >= 4 and M >= 4:
            return "Recent High-Value One-Time"

        elif R >= 4:
            return "Recent Standard One-Time"

        elif R <= 2 and M >= 4:
            return "Lapsed High-Value One-Time"

        else:
            return "Hibernating / Low Engagement"


rfm["customer_segment"] = rfm.apply(
    assign_customer_segment,
    axis=1
)

In [190]:
print(
    "Customers in RFM:",
    len(rfm)
)

print(
    "Customers with missing segment:",
    rfm["customer_segment"].isnull().sum()
)

print(
    "Number of segments:",
    rfm["customer_segment"].nunique()
)

Customers in RFM: 95420
Customers with missing segment: 0
Number of segments: 7


In [191]:
rfm["customer_segment"].value_counts()

customer_segment
Hibernating / Low Engagement    41490
Recent Standard One-Time        22237
Recent High-Value One-Time      14822
Lapsed High-Value One-Time      13958
At-Risk Repeat Buyers            1654
Champions                         974
Recent Repeat Buyers              285
Name: count, dtype: int64

In [192]:
segment_profile = (
    rfm
    .groupby("customer_segment")
    .agg(
        customers=(
            "customer_unique_id",
            "nunique"
        ),

        avg_recency=(
            "recency",
            "mean"
        ),

        avg_frequency=(
            "frequency",
            "mean"
        ),

        avg_monetary=(
            "monetary",
            "mean"
        ),

        total_monetary=(
            "monetary",
            "sum"
        )
    )
    .reset_index()
)

In [193]:
segment_profile["customer_share_pct"] = (
    segment_profile["customers"]
    / segment_profile["customers"].sum()
    * 100
)

In [194]:
segment_profile["monetary_share_pct"] = (
    segment_profile["total_monetary"]
    / segment_profile["total_monetary"].sum()
    * 100
)

In [195]:
segment_profile[
    [
        "customer_segment",
        "customers",
        "customer_share_pct",
        "avg_recency",
        "avg_frequency",
        "avg_monetary",
        "total_monetary",
        "monetary_share_pct"
    ]
].sort_values(
    "total_monetary",
    ascending=False
).round(2)

,customer_segment,customers,customer_share_pct,avg_recency,avg_frequency,avg_monetary,total_monetary,monetary_share_pct
4,Recent High-Value One-Time,14822,15.53,141.76,1.00,269.21,3990159.10,29.36
3,Lapsed High-Value One-Time,13958,14.63,446.17,1.00,279.84,3905979.49,28.74
2,Hibernating / Low Engagement,41490,43.48,368.07,1.00,89.02,3693623.64,27.18
6,Recent Standard One-Time,22237,23.30,139.61,1.00,55.70,1238589.61,9.11
0,At-Risk Repeat Buyers,1654,1.73,371.03,2.08,256.27,423866.30,3.12
1,Champions,974,1.02,139.20,2.19,326.80,318304.27,2.34
5,Recent Repeat Buyers,285,0.30,138.06,2.03,74.11,21121.29,0.16


In [196]:
segment_profile["value_index"] = (
    segment_profile["monetary_share_pct"]
    / segment_profile["customer_share_pct"]
)

In [197]:
segment_profile[
    [
        "customer_segment",
        "customers",
        "customer_share_pct",
        "avg_recency",
        "avg_frequency",
        "avg_monetary",
        "monetary_share_pct",
        "value_index"
    ]
].sort_values(
    "value_index",
    ascending=False
).round(2)

,customer_segment,customers,customer_share_pct,avg_recency,avg_frequency,avg_monetary,monetary_share_pct,value_index
1,Champions,974,1.02,139.20,2.19,326.80,2.34,2.29
3,Lapsed High-Value One-Time,13958,14.63,446.17,1.00,279.84,28.74,1.96
4,Recent High-Value One-Time,14822,15.53,141.76,1.00,269.21,29.36,1.89
0,At-Risk Repeat Buyers,1654,1.73,371.03,2.08,256.27,3.12,1.80
2,Hibernating / Low Engagement,41490,43.48,368.07,1.00,89.02,27.18,0.62
5,Recent Repeat Buyers,285,0.30,138.06,2.03,74.11,0.16,0.52
6,Recent Standard One-Time,22237,23.30,139.61,1.00,55.70,9.11,0.39


In [198]:
segment_actions = {
    "Champions":
        "Prioritize retention, loyalty rewards, VIP benefits, and personalized cross-sell opportunities.",

    "Recent Repeat Buyers":
        "Encourage continued repeat purchasing through personalized recommendations and loyalty incentives.",

    "At-Risk Repeat Buyers":
        "Use targeted reactivation campaigns, personalized offers, and win-back messaging.",

    "Recent High-Value One-Time":
        "Prioritize second-purchase conversion with relevant recommendations and time-sensitive incentives.",

    "Recent Standard One-Time":
        "Drive a second purchase through onboarding, recommendations, and targeted follow-up campaigns.",

    "Lapsed High-Value One-Time":
        "Run high-value win-back campaigns based on previous purchasing behavior.",

    "Hibernating / Low Engagement":
        "Use cost-efficient re-engagement campaigns and avoid excessive promotional spend."
}

segment_profile["recommended_action"] = (
    segment_profile["customer_segment"]
    .map(segment_actions)
)

In [199]:
segment_profile[
    [
        "customer_segment",
        "customers",
        "customer_share_pct",
        "monetary_share_pct",
        "value_index",
        "recommended_action"
    ]
].sort_values(
    "value_index",
    ascending=False
)

,customer_segment,customers,customer_share_pct,monetary_share_pct,value_index,recommended_action
1,Champions,974,1.020750,2.341912,2.294304,"Prioritize retention, loyalty rewards, VIP ben..."
3,Lapsed High-Value One-Time,13958,14.627961,28.738095,1.964600,Run high-value win-back campaigns based on pre...
4,Recent High-Value One-Time,14822,15.533431,29.357443,1.889952,Prioritize second-purchase conversion with rel...
0,At-Risk Repeat Buyers,1654,1.733389,3.118580,1.799123,"Use targeted reactivation campaigns, personali..."
2,Hibernating / Low Engagement,41490,43.481450,27.175695,0.624995,Use cost-efficient re-engagement campaigns and...
5,Recent Repeat Buyers,285,0.298680,0.155399,0.520287,Encourage continued repeat purchasing through ...
6,Recent Standard One-Time,22237,23.304339,9.112876,0.391038,"Drive a second purchase through onboarding, re..."


### RFM Customer Segmentation Findings

Customer-level RFM analysis identified seven behavioral segments based on recency, purchase frequency, and merchandise value.

Key findings:

- Repeat purchasing was limited within the observed dataset period, making one-time customer behavior particularly important for retention analysis.

- Champions represented approximately 1.02% of customers and 2.34% of merchandise value, with a value index of 2.29, indicating disproportionately high customer value relative to segment size.

- Recent High-Value One-Time Buyers represented 15.53% of customers but approximately 29.36% of merchandise value.

- Lapsed High-Value One-Time Buyers represented 14.63% of customers and approximately 28.74% of merchandise value.

- Combined, the two high-value one-time segments represented approximately 30.16% of customers and 58.10% of merchandise value. Because these segments are partly defined using monetary scores, their high monetary contribution is expected; however, their one-time purchase behavior highlights a potentially important second-purchase conversion and reactivation opportunity.

- Hibernating / Low Engagement customers formed the largest segment at 43.48% of customers but had a value index of only 0.62, suggesting lower monetary contribution relative to segment size.

The segmentation suggests that customer strategy should differentiate between retention of established repeat buyers, second-purchase conversion of recent high-value customers, reactivation of lapsed high-value customers, and cost-efficient engagement of lower-value inactive customers.

In [200]:
cohort_source = customer_orders[
    [
        "customer_unique_id",
        "order_id",
        "order_purchase_timestamp"
    ]
].copy()

cohort_source.head()

,customer_unique_id,order_id,order_purchase_timestamp
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33
1,af07308b275d755c9edb36a90c618231,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37
2,3a653a41f6f9fc3d2a113cf8398680e8,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49
3,7c142cf63193a1473d2e66489a9ae977,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06
4,72632f0f9dd73dfee390c9b22eb56dd6,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39


In [201]:
print(
    "Missing customer IDs:",
    cohort_source["customer_unique_id"].isnull().sum()
)

print(
    "Missing purchase dates:",
    cohort_source["order_purchase_timestamp"].isnull().sum()
)

Missing customer IDs: 0
Missing purchase dates: 0


In [202]:
cohort_source["order_month"] = (
    cohort_source[
        "order_purchase_timestamp"
    ].dt.to_period("M")
)

In [203]:
cohort_source[
    [
        "order_purchase_timestamp",
        "order_month"
    ]
].head()

,order_purchase_timestamp,order_month
0,2017-10-02 10:56:33,2017-10
1,2018-07-24 20:41:37,2018-07
2,2018-08-08 08:38:49,2018-08
3,2017-11-18 19:28:06,2017-11
4,2018-02-13 21:18:39,2018-02


In [204]:
cohort_source["cohort_month"] = (
    cohort_source
    .groupby("customer_unique_id")[
        "order_month"
    ]
    .transform("min")
)

In [205]:
cohort_source[
    [
        "customer_unique_id",
        "order_month",
        "cohort_month"
    ]
].head(10)

,customer_unique_id,order_month,cohort_month
0,7c396fd4830fd04220f754e42b4e5bff,2017-10,2017-09
1,af07308b275d755c9edb36a90c618231,2018-07,2018-07
2,3a653a41f6f9fc3d2a113cf8398680e8,2018-08,2018-08
3,7c142cf63193a1473d2e66489a9ae977,2017-11,2017-11
4,72632f0f9dd73dfee390c9b22eb56dd6,2018-02,2018-02
5,80bb27c7c16e8f973207a5086ab329e2,2017-07,2017-07
6,36edbb3fb164b1f16485364b6fb04c73,2017-04,2017-04
7,932afa1e708222e5821dac9cd5db4cae,2017-05,2017-05
8,39382392765b6dc74812866ee5ee92a7,2017-01,2017-01
9,299905e3934e9e181bfb2e164dd4b4f8,2017-07,2017-07


In [206]:
cohort_source["cohort_index"] = (
    (
        cohort_source["order_month"].dt.year
        - cohort_source["cohort_month"].dt.year
    ) * 12
    +
    (
        cohort_source["order_month"].dt.month
        - cohort_source["cohort_month"].dt.month
    )
)

In [207]:
cohort_source[
    [
        "customer_unique_id",
        "order_month",
        "cohort_month",
        "cohort_index"
    ]
].sort_values(
    [
        "customer_unique_id",
        "order_month"
    ]
).head(20)

,customer_unique_id,order_month,cohort_month,cohort_index
52798,0000366f3b9a7992bf8c76cfdf3221e2,2018-05,2018-05,0
73889,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05,2018-05,0
26460,0000f46a3911fa3c0805444483337064,2017-03,2017-03,0
98493,0000f6ccb0745a6a4b88665a16c9f078,2017-10,2017-10,0
41564,0004aac84e0df4da2b147fca70cf8255,2017-11,2017-11,0
78514,0004bd2a26a76fe21f786e4fbd80607f,2018-04,2018-04,0
74715,00050ab1314c0e55a6ca13cf7181fecf,2018-04,2018-04,0
6867,00053a61a98854899e70ed204dd4bafe,2018-02,2018-02,0
71235,0005e1862207bf6ccc02e4228effd9a0,2017-03,2017-03,0
68876,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03,2018-03,0


In [208]:
cohort_counts = (
    cohort_source
    .groupby(
        [
            "cohort_month",
            "cohort_index"
        ]
    )
    .agg(
        active_customers=(
            "customer_unique_id",
            "nunique"
        )
    )
    .reset_index()
)

cohort_counts.head(20)

,cohort_month,cohort_index,active_customers
0,2016-09,0,4
1,2016-10,0,321
2,2016-10,6,1
3,2016-10,9,1
4,2016-10,11,1
5,2016-10,13,1
6,2016-10,15,1
7,2016-10,17,1
8,2016-10,19,2
9,2016-10,20,2


In [209]:
cohort_matrix = (
    cohort_counts
    .pivot(
        index="cohort_month",
        columns="cohort_index",
        values="active_customers"
    )
)

cohort_matrix.head()

cohort_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,321.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,NaN,1.0,NaN,1.0,NaN,1.0,NaN,1.0,2.0,2.0
2016-12,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,764.0,3.0,2.0,1.0,3.0,1.0,4.0,1.0,1.0,NaN,3.0,1.0,6.0,3.0,1.0,1.0,2.0,3.0,1.0,NaN
2017-02,1752.0,4.0,5.0,2.0,7.0,2.0,4.0,3.0,3.0,4.0,2.0,5.0,3.0,3.0,2.0,1.0,1.0,4.0,NaN,NaN


In [210]:
cohort_sizes = cohort_matrix[0]

In [211]:
retention_matrix = (
    cohort_matrix
    .divide(
        cohort_sizes,
        axis=0
    )
    * 100
)

retention_matrix.round(2).head()

cohort_index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-10,100.0,NaN,NaN,NaN,NaN,NaN,0.31,NaN,NaN,0.31,NaN,0.31,NaN,0.31,NaN,0.31,NaN,0.31,0.62,0.62
2016-12,100.0,100.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-01,100.0,0.39,0.26,0.13,0.39,0.13,0.52,0.13,0.13,NaN,0.39,0.13,0.79,0.39,0.13,0.13,0.26,0.39,0.13,NaN
2017-02,100.0,0.23,0.29,0.11,0.40,0.11,0.23,0.17,0.17,0.23,0.11,0.29,0.17,0.17,0.11,0.06,0.06,0.23,NaN,NaN


In [212]:
cohort_size_summary = (
    cohort_sizes
    .reset_index()
)

cohort_size_summary.columns = [
    "cohort_month",
    "new_customers"
]

cohort_size_summary.sort_values(
    "cohort_month"
)

,cohort_month,new_customers
0,2016-09,4.0
1,2016-10,321.0
2,2016-12,1.0
3,2017-01,764.0
4,2017-02,1752.0
5,2017-03,2636.0
6,2017-04,2352.0
7,2017-05,3596.0
8,2017-06,3139.0
9,2017-07,3894.0


In [213]:
month_1_counts = (
    cohort_counts[
        cohort_counts["cohort_index"] == 1
    ][
        [
            "cohort_month",
            "active_customers"
        ]
    ]
    .rename(
        columns={
            "active_customers":
            "month_1_active"
        }
    )
)

In [214]:
month_1_retention = (
    cohort_size_summary
    .merge(
        month_1_counts,
        on="cohort_month",
        how="left"
    )
)

month_1_retention[
    "month_1_active"
] = (
    month_1_retention[
        "month_1_active"
    ].fillna(0)
)

In [215]:
last_observed_month = (
    cohort_source["order_month"].max()
)

eligible_month_1 = (
    month_1_retention[
        month_1_retention[
            "cohort_month"
        ] < last_observed_month
    ]
)

In [216]:
overall_month_1_retention = (
    eligible_month_1[
        "month_1_active"
    ].sum()
    /
    eligible_month_1[
        "new_customers"
    ].sum()
    * 100
)

print(
    "Weighted Month-1 Retention Rate:",
    round(
        overall_month_1_retention,
        2
    ),
    "%"
)

Weighted Month-1 Retention Rate: 0.48 %


In [217]:
def calculate_weighted_retention(
    cohort_counts,
    cohort_size_summary,
    target_month,
    last_observed_month
):

    target_counts = (
        cohort_counts[
            cohort_counts[
                "cohort_index"
            ] == target_month
        ][
            [
                "cohort_month",
                "active_customers"
            ]
        ]
        .rename(
            columns={
                "active_customers":
                "retained_customers"
            }
        )
    )

    retention_data = (
        cohort_size_summary
        .merge(
            target_counts,
            on="cohort_month",
            how="left"
        )
    )

    retention_data[
        "retained_customers"
    ] = (
        retention_data[
            "retained_customers"
        ].fillna(0)
    )

    eligible = retention_data[
        (
            last_observed_month
            - retention_data["cohort_month"]
        ).apply(
            lambda x: x.n
        ) >= target_month
    ]

    rate = (
        eligible[
            "retained_customers"
        ].sum()
        /
        eligible[
            "new_customers"
        ].sum()
        * 100
    )

    return rate

In [218]:
month_1 = calculate_weighted_retention(
    cohort_counts,
    cohort_size_summary,
    1,
    last_observed_month
)

month_3 = calculate_weighted_retention(
    cohort_counts,
    cohort_size_summary,
    3,
    last_observed_month
)

month_6 = calculate_weighted_retention(
    cohort_counts,
    cohort_size_summary,
    6,
    last_observed_month
)

print(
    f"Month-1 Retention: {month_1:.2f}%"
)

print(
    f"Month-3 Retention: {month_3:.2f}%"
)

print(
    f"Month-6 Retention: {month_6:.2f}%"
)

Month-1 Retention: 0.48%
Month-3 Retention: 0.22%
Month-6 Retention: 0.19%


In [219]:
print(
    "Minimum cohort index:",
    cohort_source[
        "cohort_index"
    ].min()
)

print(
    "Maximum cohort index:",
    cohort_source[
        "cohort_index"
    ].max()
)

print(
    "Number of negative cohort indices:",
    (
        cohort_source[
            "cohort_index"
        ] < 0
    ).sum()
)

Minimum cohort index: 0
Maximum cohort index: 20
Number of negative cohort indices: 0


Raw Data
    ↓
Data Profiling
    ↓
Data Quality Assessment
    ↓
Primary Key Validation
    ↓
Grain Validation
    ↓
Chronological Validation
    ↓
Metric-specific Cleaning Rules
    ↓
Financial Reconciliation
    ↓
Customer Segmentation
    ↓
Cohort Retention